# Data Mining Lab 1: Spark Data Lake Preprocessing

**Dataset:** `sales.csv` | **Environment:** Google Colab / PySpark

**Student Name:** Tanishk Khandelwal | **Roll Number:** LIT2023020

---

This notebook implements a two-layer Data Lake architecture:
- **Bronze Layer** — raw CSV preserved exactly as provided
- **Silver Layer** — cleaned and preprocessed using Apache Spark

All data quality checks (duplicates, missing values, inconsistencies) are
performed through Spark operations.

## 1. Install PySpark & Create SparkSession

Google Colab provides Java out of the box. We install PySpark and
create a local SparkSession for processing.

In [1]:
%pip -q install "pyspark==4.0.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, shutil

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("DataMiningLab1_LIT2023020")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.1


## 2. Set Up Data Lake Folders & Download Dataset to Bronze

The Bronze layer stores the original CSV byte-for-byte. We download it
from the class repository and verify its SHA-256 checksum for reproducibility.

```
data_lake/
  bronze/sales.csv          ← raw, untouched
  silver/preprocessed_sales/ ← Spark CSV output
silver/sales_silver.csv     ← single-file copy for submission
```

In [3]:
ROOT = Path.cwd()
BRONZE = ROOT / "data_lake" / "bronze"
SILVER = ROOT / "data_lake" / "silver" / "preprocessed_sales"
SILVER_SUBMIT = ROOT / "silver"

BRONZE.mkdir(parents=True, exist_ok=True)
SILVER.parent.mkdir(parents=True, exist_ok=True)
SILVER_SUBMIT.mkdir(parents=True, exist_ok=True)

# Pinned revision for reproducibility
REVISION = "81539d8aee71b8ce4c428365cea26e6b272723c6"
CSV_URL = (
    "https://raw.githubusercontent.com/necromancersanju-netizen/"
    f"data-mining-data-lake-assignment/{REVISION}/dataset/sales.csv"
)
EXPECTED_HASH = "1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8"

bronze_csv = BRONZE / "sales.csv"
if not bronze_csv.exists():
    with urlopen(CSV_URL, timeout=60) as resp:
        raw_bytes = resp.read()
    assert hashlib.sha256(raw_bytes).hexdigest() == EXPECTED_HASH, "Checksum mismatch!"
    bronze_csv.write_bytes(raw_bytes)

bronze_sha = hashlib.sha256(bronze_csv.read_bytes()).hexdigest()
assert bronze_sha == EXPECTED_HASH, "Bronze file has been altered!"
print("Bronze CSV downloaded and verified.")
print("SHA-256:", bronze_sha)

Bronze CSV downloaded and verified.
SHA-256: 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8


## 3. Load & Inspect Raw Data

We read all columns as strings (no schema inference) to preserve the
original values, including invalid ones, for inspection.

In [4]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("ignoreLeadingWhiteSpace", False)
    .option("ignoreTrailingWhiteSpace", False)
    .option("mode", "FAILFAST")
    .csv(str(bronze_csv))
    .cache()
)

columns = raw_df.columns
total_before = raw_df.count()

raw_df.printSchema()
raw_df.show(8, truncate=False)
print("Total Bronze records:", total_before)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+--

## 4. Identify Duplicates

Check for exact duplicate rows and repeated `order_id` values.

In [5]:
dedup_df = raw_df.dropDuplicates().cache()
after_dedup = dedup_df.count()
n_duplicates = total_before - after_dedup

print(f"Exact duplicate rows found: {n_duplicates}")
print(f"Rows after removing duplicates: {after_dedup}")

# Show which rows were duplicated
raw_df.groupBy(*columns).count().filter(F.col("count") > 1).show(5, truncate=False)

# Check for repeated order IDs (different data, same ID)
repeated_ids = dedup_df.groupBy("order_id").count().filter(F.col("count") > 1)
print(f"Order IDs appearing in multiple distinct rows: {repeated_ids.count()}")
repeated_ids.orderBy("order_id").show(truncate=False)

Exact duplicate rows found: 10
Rows after removing duplicates: 990
+--------+-----------+--------------+---------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name |product        |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+--------------+---------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100444  |C0090      |Abhinav Pillai|Novel          |Books    |8       |530       |15              |Credit Card     |Mumbai    |Maharashtra   |2025-09-11|Delivered   |2    |
|100773  |C0315      |Meera Malhotra|Study Table    |Furniture|7       |11310     |40              |Debit Card      |Pune      |Maharashtra   |2026-02-25|Delivered   |2    |
|100297  |C0149      |Manish Pillai |Dry Fruits Pack|Grocery  |

## 5. Identify Missing Values & Text Inconsistencies

Missing = null or whitespace-only. We also check for capitalization
inconsistencies in text columns (e.g. `CLOTHING` vs `Clothing`).

In [6]:
# Helper: detect missing (null or blank)
def is_blank(col_name):
    c = F.col(col_name)
    return c.isNull() | (F.trim(c) == "")

# Count missing values per column
missing_counts = raw_df.select([
    F.count(F.when(is_blank(c), 1)).alias(c) for c in columns
]).first().asDict()

print("Missing values per column:")
for col_name in columns:
    ct = missing_counts[col_name]
    marker = " ⚠" if ct > 0 else ""
    print(f"  {col_name:20s} {ct}{marker}")

Missing values per column:
  order_id             0
  customer_id          0
  customer_name        10 ⚠
  product              0
  category             0
  quantity             0
  unit_price           0
  discount_percent     0
  payment_method       8 ⚠
  city                 8 ⚠
  state                0
  order_date           0
  order_status         0


In [7]:
# Check text inconsistencies — show distinct values that differ from title case
text_cols = ["category", "city", "customer_name"]

for tc in text_cols:
    variants = (
        raw_df
        .filter(~is_blank(tc))
        .filter(F.col(tc) != F.initcap(F.trim(F.col(tc))))
        .select(tc).distinct()
    )
    n = variants.count()
    if n > 0:
        print(f"\n'{tc}' — {n} values needing standardization:")
        variants.show(truncate=False)


'category' — 5 values needing standardization:
+-----------+
|category   |
+-----------+
|ELECTRONICS|
|CLOTHING   |
|sports     |
|beauty     |
|SPORTS     |
+-----------+


'city' — 7 values needing standardization:
+-----------+
|city       |
+-----------+
| Mumbai    |
|coimbatore |
|HYDERABAD  |
| Hyderabad |
|JAIPUR     |
|delhi      |
|DELHI      |
+-----------+


'customer_name' — 3 values needing standardization:
+----------------+
|customer_name   |
+----------------+
|Tanvi Chaudhary |
|Meera Joshi     |
|Akash Jain      |
+----------------+



## 6. Identify Invalid Numbers & Dates

**Rules:**
- `quantity` — must be a positive integer
- `unit_price` — must be positive
- `discount_percent` — must be between 0 and 100
- `order_date` — must be a real calendar date; accepted formats:
  `yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy`, `MM-dd-yyyy`

Invalid values are set to null (not dropped) and flagged.

In [8]:
# Multi-format date parser using try_to_timestamp
DATE_FMTS = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "MM-dd-yyyy"]

def safe_parse_date(col_name):
    return F.coalesce(*[
        F.try_to_timestamp(F.trim(F.col(col_name)), F.lit(fmt))
        for fmt in DATE_FMTS
    ]).cast("date")

# Add parsed helper columns
def add_parsed_cols(df):
    return (
        df
        .withColumn("_qty", F.trim(F.col("quantity")).try_cast("double"))
        .withColumn("_price", F.trim(F.col("unit_price")).try_cast("decimal(12,2)"))
        .withColumn("_disc", F.trim(F.col("discount_percent")).try_cast("double"))
        .withColumn("_date", safe_parse_date("order_date"))
    )

# Validity conditions
qty_ok = (
    F.col("_qty").isNotNull() & (~F.isnan("_qty"))
    & (F.col("_qty") > 0)
    & (F.col("_qty") == F.floor("_qty"))
)
price_ok = F.col("_price").isNotNull() & (F.col("_price") > 0)
disc_ok = (
    F.col("_disc").isNotNull() & (~F.isnan("_disc"))
    & F.col("_disc").between(0, 100)
)
date_ok = F.col("_date").isNotNull()

# Count invalids
raw_parsed = add_parsed_cols(raw_df)
invalid_counts = raw_parsed.select(
    F.count(F.when(~qty_ok, 1)).alias("invalid_quantity"),
    F.count(F.when(~price_ok, 1)).alias("invalid_unit_price"),
    F.count(F.when(~disc_ok, 1)).alias("invalid_discount_percent"),
    F.count(F.when(~date_ok, 1)).alias("invalid_order_date"),
).first().asDict()

print("Invalid/missing numeric & date values:")
for k, v in invalid_counts.items():
    print(f"  {k:30s} {v}")

Invalid/missing numeric & date values:
  invalid_quantity               12
  invalid_unit_price             8
  invalid_discount_percent       8
  invalid_order_date             4


In [9]:
# Show problematic date values
print("Date values that are not in standard yyyy-MM-dd format or are invalid:")
raw_parsed.filter(
    ~date_ok | (
        date_ok & (F.trim(F.col("order_date")) != F.date_format("_date", "yyyy-MM-dd"))
    )
).select("order_id", "order_date", "_date").show(truncate=False)

Date values that are not in standard yyyy-MM-dd format or are invalid:
+--------+----------+----------+
|order_id|order_date|_date     |
+--------+----------+----------+
|100097  |31/02/2026|NULL      |
|100358  |2025/07/14|2025-07-14|
|100448  |not-a-date|NULL      |
|100634  |2025-02-30|NULL      |
|100637  |2026-13-05|NULL      |
|100969  |09-15-2026|2026-09-15|
+--------+----------+----------+



## 7. Preprocessing — Build the Silver Layer

Starting from deduplicated data, we apply the following transformations:

1. **Remove exact duplicate rows** (already done in step 4)
2. **Fill missing text** with `"Unknown"`
3. **Standardize text** — `initcap()` for names/category/city/state/status;
   preserve `UPI` and `Cash on Delivery` in payment_method
4. **Validate numbers** — invalid quantity/price/discount → null
5. **Standardize dates** — parse multiple formats into `yyyy-MM-dd`;
   invalid dates → null
6. **Add `data_quality_issues` flag** for rows with remaining nulls

Rows are **kept** even when some fields are invalid — the quality flag
indicates what's wrong.

In [10]:
# Columns whose missing text should become "Unknown"
text_fill_cols = [
    "customer_name", "product", "category",
    "payment_method", "city", "state", "order_status",
]

# Columns to title-case
title_cols = {"customer_name", "category", "city", "state", "order_status"}

def normalize_text(col_name):
    """Return normalized expression for a text column."""
    trimmed = F.trim(F.col(col_name))
    if col_name in title_cols:
        return F.initcap(trimmed)
    if col_name == "payment_method":
        return (
            F.when(F.upper(trimmed) == "UPI", F.lit("UPI"))
            .when(F.lower(trimmed) == "cash on delivery", F.lit("Cash on Delivery"))
            .otherwise(F.initcap(trimmed))
        )
    return trimmed  # product — preserve original casing

# --- Build quality issue conditions ---
issue_labels = {}
for c in text_fill_cols:
    issue_labels[f"missing_{c}"] = is_blank(c)
issue_labels["invalid_or_missing_quantity"] = ~qty_ok
issue_labels["invalid_or_missing_unit_price"] = ~price_ok
issue_labels["invalid_or_missing_discount_percent"] = ~disc_ok
issue_labels["invalid_or_missing_order_date"] = ~date_ok

# Start from deduplicated + parsed data
working = add_parsed_cols(dedup_df)

# Build the data_quality_issues column
working = working.withColumn(
    "data_quality_issues",
    F.concat_ws("; ", *[
        F.when(cond, F.lit(label)) for label, cond in issue_labels.items()
    ])
)
working = working.withColumn(
    "data_quality_issues",
    F.when(F.col("data_quality_issues") == "", F.lit("none"))
    .otherwise(F.col("data_quality_issues"))
)

# Apply text normalization + fill missing
for c in text_fill_cols:
    working = working.withColumn(
        c,
        F.when(is_blank(c), F.lit("Unknown"))
        .otherwise(normalize_text(c))
    )

# Apply numeric + date transformations
silver_df = (
    working
    .withColumn("quantity", F.when(qty_ok, F.col("_qty").cast("int")))
    .withColumn("unit_price", F.when(price_ok, F.col("_price")))
    .withColumn("discount_percent", F.when(disc_ok, F.col("_disc").cast("decimal(5,2)")))
    .withColumn("order_date", F.col("_date"))
    .select(*columns, "data_quality_issues")
    .cache()
)

silver_count = silver_df.count()
flagged_count = silver_df.filter(F.col("data_quality_issues") != "none").count()

print(f"Bronze records:              {total_before}")
print(f"Duplicates removed:          {n_duplicates}")
print(f"Silver records:              {silver_count}")
print(f"Rows with quality flags:     {flagged_count}")

silver_df.printSchema()
silver_df.orderBy("order_id").show(8, truncate=False)

Bronze records:              1000
Duplicates removed:          10
Silver records:              990
Rows with quality flags:     58
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount_percent: decimal(5,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- data_quality_issues: string (nullable = false)

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-------------------+
|order_id|customer_id|customer_name  |product      |category |quantity

In [11]:
# Show flagged rows
print("Examples of rows retained with quality flags:")
silver_df.filter(F.col("data_quality_issues") != "none").select(
    "order_id", "quantity", "unit_price", "discount_percent",
    "order_date", "data_quality_issues"
).orderBy("order_id").show(12, truncate=False)

Examples of rows retained with quality flags:
+--------+--------+----------+----------------+----------+-----------------------------------+
|order_id|quantity|unit_price|discount_percent|order_date|data_quality_issues                |
+--------+--------+----------+----------------+----------+-----------------------------------+
|100013  |3       |660.00    |12.00           |2025-08-16|missing_customer_name              |
|100034  |7       |1720.00   |NULL            |2025-08-19|invalid_or_missing_discount_percent|
|100051  |7       |710.00    |15.00           |2026-07-02|missing_payment_method             |
|100097  |4       |47430.00  |10.00           |NULL      |invalid_or_missing_order_date      |
|100102  |7       |NULL      |40.00           |2025-03-02|invalid_or_missing_unit_price      |
|100114  |NULL    |2920.00   |15.00           |2025-02-15|invalid_or_missing_quantity        |
|100140  |4       |790.00    |NULL            |2025-10-01|invalid_or_missing_discount_percent|
|100

## 8. Save Silver Layer

Write the processed DataFrame to the Silver folder in CSV format.
Since this is a small dataset, `coalesce(1)` writes a single part file.
We copy that file to `silver/sales_silver.csv` for submission.

In [12]:
(
    silver_df.orderBy(*columns).coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .option("nullValue", "")
    .csv(str(SILVER))
)

part_files = list(SILVER.glob("part-*.csv"))
assert len(part_files) == 1, f"Expected 1 part file, got {len(part_files)}"

submission_csv = SILVER_SUBMIT / "sales_silver.csv"
shutil.copyfile(part_files[0], submission_csv)

print("Spark output directory:", SILVER)
print("Submission CSV:", submission_csv)

Spark output directory: /content/data_lake/silver/preprocessed_sales
Submission CSV: /content/silver/sales_silver.csv


## 9. Verification — Read Back & Validate

Read the saved Silver CSV back into Spark to confirm row counts,
schema, and remaining null counts.

In [13]:
# Define the expected schema for reading back
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DecimalType, DateType
)

silver_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("customer_name", StringType()),
    StructField("product", StringType()),
    StructField("category", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DecimalType(12, 2)),
    StructField("discount_percent", DecimalType(5, 2)),
    StructField("payment_method", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("order_date", DateType()),
    StructField("order_status", StringType()),
    StructField("data_quality_issues", StringType()),
])

readback = (
    spark.read.option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .schema(silver_schema)
    .csv(str(SILVER))
    .cache()
)

rb_count = readback.count()
rb_flagged = readback.filter(F.col("data_quality_issues") != "none").count()

print("=" * 55)
print("VERIFICATION SUMMARY")
print("=" * 55)
print(f"Bronze rows (before preprocessing):   {total_before}")
print(f"Exact duplicates removed:             {n_duplicates}")
print(f"Silver rows (after preprocessing):    {silver_count}")
print(f"Silver rows read back:                {rb_count}")
print(f"Rows with quality flags:              {rb_flagged}")
assert rb_count == silver_count, "Row count mismatch!"
print("\n✓ Row counts match.")

VERIFICATION SUMMARY
Bronze rows (before preprocessing):   1000
Exact duplicates removed:             10
Silver rows (after preprocessing):    990
Silver rows read back:                990
Rows with quality flags:              58

✓ Row counts match.


In [14]:
# Remaining nulls in Silver (intentional — invalid values set to null)
print("Remaining nulls in Silver (expected for invalid/missing values):")
null_counts = readback.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in silver_schema.fieldNames()
]).first().asDict()

for col_name in silver_schema.fieldNames():
    ct = null_counts[col_name]
    print(f"  {col_name:20s} {ct}")

Remaining nulls in Silver (expected for invalid/missing values):
  order_id             0
  customer_id          0
  customer_name        0
  product              0
  category             0
  quantity             12
  unit_price           8
  discount_percent     8
  payment_method       0
  city                 0
  state                0
  order_date           4
  order_status         0
  data_quality_issues  0


In [15]:
# Display a few Silver rows
print("Sample Silver rows:")
readback.orderBy("order_id").show(10, truncate=False)

Sample Silver rows:
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-------------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|data_quality_issues|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-------------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710.00   |20.00           |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |none               |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680.00    |5.00            |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |none               |
|1000

In [16]:
# Confirm Bronze file is untouched
bronze_sha_after = hashlib.sha256(bronze_csv.read_bytes()).hexdigest()
assert bronze_sha_after == EXPECTED_HASH, "Bronze was modified!"
print("✓ Bronze file unchanged — SHA-256:", bronze_sha_after)

✓ Bronze file unchanged — SHA-256: 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8


## 10. Preprocessing Summary

| Step | Operation | Details |
|------|-----------|--------|
| 1 | **Remove exact duplicates** | 10 identical rows removed → 990 unique rows |
| 2 | **Fill missing text** | `customer_name` (10), `payment_method` (8), `city` (8) filled with `"Unknown"` |
| 3 | **Standardize text** | Title-cased names, categories, cities, states, status; preserved `UPI` and `Cash on Delivery` |
| 4 | **Validate quantity** | 12 invalid/missing → set to null |
| 5 | **Validate unit_price** | 8 invalid/missing → set to null |
| 6 | **Validate discount** | 8 invalid/missing → set to null |
| 7 | **Standardize dates** | Parsed 4 formats into `yyyy-MM-dd`; 4 invalid dates (e.g., `31/02/2026`, `not-a-date`) → null |
| 8 | **Quality flag** | Added `data_quality_issues` column; 58 rows flagged |

All discrepancy counts were obtained through **Spark operations** — no
hard-coded values. Rows with issues are **retained** (not dropped) with
quality flags for downstream consumers.

In [17]:
spark.stop()
print("SparkSession stopped. Done!")

SparkSession stopped. Done!
